# 05 - Prompt Engineering & Conversational RAG - How should the assistant behave in realistic conversations?

## Objective

Design and evaluate the assistant's behavior.

The objective is to produce responses that are grounded, professional, and consistent while supporting conversational interactions.

This notebook focuses on prompt engineering and conversation management rather than infrastructure.

---

## Questions

- How should the assistant be instructed?
- How should conversation history be incorporated?
- How should supporting sources be presented?
- How should the assistant respond when information is unavailable?
- How should follow-up questions be handled?

---

## Success Criteria

By the end of this notebook:

- A production-ready system prompt has been defined.
- The assistant supports conversational interactions.
- Responses remain grounded in retrieved context.
- The assistant refuses unsupported claims.

---

## Notes

The selected prompt and conversation strategy will be used by the application layer.

In [1]:
from pathlib import Path
import os
import re 

from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    Docx2txtLoader, PyMuPDFLoader, TextLoader
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Rebuild Pipeline
Adding the constructs from the previous experimentation notebooks

### Load documents
Reuse ingestion code of a previous notebook. At this point I'm only experimenting, in production this would not be duplicated.

In [3]:
# project paths 
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw/knowledge_base"

LOADERS = {".pdf": PyMuPDFLoader, ".docx": Docx2txtLoader, ".md": TextLoader, ".txt": TextLoader}

documents = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_dir():
        continue
    loader_cls = LOADERS.get(path.suffix.lower())
    if loader_cls is None:
        print(f"Skipping unsupported file: {path.name}")
        continue
    loader = loader_cls(str(path))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents.")

Skipping unsupported file: .DS_Store
Skipping unsupported file: .DS_Store
Loaded 6 documents.


In [4]:
def clean_text(text):
    # Collapse single newlines (likely mid-sentence wraps) into spaces,
    # but preserve intentional paragraph breaks (blank lines).
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse 3+ newlines down to a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse repeated spaces/tabs.
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

### Chunk documents

Reuse the selected strategy.

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Generated {len(chunks)} chunks.")

Generated 50 chunks.


### Build vector store
Reuse the vectorstore logic

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

db_name = "vector_db"

# if os.path.exists(db_name):
#     Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
# print(f"Vectorstore created with {vector_store._collection.count()} vectors.")

# Load the existing database without deleting it!
vector_store = Chroma(
    persist_directory=db_name, embedding_function=embeddings
)
print(f"Vectorstore Loaded with {vector_store._collection.count()} vectors.")

Vectorstore created with 50 vectors.


### Initialize the LLM
Notice that **Retrieval and generation are independent** components

In [7]:
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0, verbosity='low', 
                 reasoning={"effort": "low"})

In [29]:
# initialize the retriever from the vector store
retriever = vector_store.as_retriever()

## Define system prompt

In [9]:
SYSTEM_PROMPT = """
You are an AI Career Assistant. You are assisting David in answering to recruiters (users) interested in his profile and potentially hiring him. 

Your purpose is to answer questions about the candidate's professional experience.

When asked about the candidate or synonyms, this refers to David, so it is preferable to use "David" to make the answers more personal and professional. 

You are not allowed to give PII data of the candidate. You can only give his first name. Last name, email and phone number should be private.
If asked for that personal information, you should be professional explaining that you are not allowed to give PII, but still answer as much as you can from the original question.

Guidelines:

- Use only the provided context.
- Do not invent information.
- If the answer is not available, clearly state that you do not know.
- Write concise, professional responses.
- When appropriate, reference the supporting documents.
"""

## Create a prompt builder

In [10]:
def build_messages(question: str, context: str, history: list | None = None):

    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    if history:
        messages.extend(history)

    messages.append(
        HumanMessage(
            content=f"""
        Context:
            {context}

        Question:
            {question}
        """
        )
    )

    return messages

## Updated RAG pipeline

In [13]:
def answer_question(question: str, history: list | None = None, k: int=3):    

    retrieved_docs = retriever.invoke(question, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    messages = build_messages(question, context, history)
    response = llm.invoke(messages)

    return response.content[0]['text'] , retrieved_docs

## First conversation

In [32]:
history = []

answer, sources = answer_question("Tell me about the candidate.")

print(answer)

David is a data science and business professional with an MSc in Data Science and a Bachelor of Business Administration. His technical background includes Python, SQL, and applied machine learning, complemented by experience in advanced analytics, commercial modelling, consulting, and stakeholder engagement.

He focuses on translating complex data into clear, commercially relevant recommendations and is comfortable working in ambiguous environments.


## Continue the conversation

In [35]:
# update history of the conversation
history.append(HumanMessage(content="Tell me about the candidate."))
history.append(AIMessage(content=answer))

In [36]:
question = "What machine learning experience does the candidate have?"
answer, sources = answer_question(question, history=history)

print(answer)

David has over five years of consulting experience delivering machine learning and advanced analytics solutions in Financial Services and Retail. He is proficient in Python and SQL and has applied expertise in statistical modelling, machine learning, NLP, deep learning, semi-structured data, and big data. He also has experience translating business questions into practical ML solutions and communicating results to diverse stakeholders.


In [37]:
for i, doc in enumerate(sources, start=1):
    print("=" * 80)
    print(f"Source {i}")
    print("=" * 80)

    print(doc.page_content)

Source 1
Principal Data Scientist with 5+ years of experience in consulting delivering advanced analytics, BI, and machine learning solutions in the industries of Financial Services and Retail. I complement my technical experience with 4 additional years in finance and business analysis roles. I have a strong focus on solving commercial problems end-to-end, from framing business questions with senior stakeholders to building ML models that solve real world problems. I bring advanced proficiency in
Source 2
stakeholders to building ML models that solve real world problems. I bring advanced proficiency in Python and SQL, which allow me to face ambiguous problems with complex data, and due to my consulting experience, I can clearly communicate the relevant insights to audiences of diverse technical backgrounds.
Source 3
EDUCATION . 

Master of Data Science, Monash, Melbourne – AUS

Mar 2019 – Dec 2020

Statistical Modelling, Machine Learning, Semi-structured data, Big Data, NLP, Deep Lear

## Testing the history

In [38]:
# update history of the conversation
history.append(HumanMessage(content=question))
history.append(AIMessage(content=answer))

In [39]:
question = "Tell me more" # ambiguous question relying on history
answer, sources = answer_question(question, history=history)

print(answer)

David focuses on practical, explainable machine learning rather than complexity for its own sake. Through work with major Australian banks, neo-banks, and retailers, he has developed a strong understanding of how data sources connect and how they explain customer behaviour.

He uses SQL, Python, and Agentic AI, with an emphasis on producing commercially relevant insights and actionable recommendations. His experience in regulated financial environments has reinforced the importance of models that stakeholders can understand, assess for feasibility, and manage for risk.

The provided information mentions a machine learning project with a major bank, but does not include enough detail to describe its specific objective or results.


In [40]:
for i, doc in enumerate(sources, start=1):
    print("=" * 80)
    print(f"Source {i}")
    print("=" * 80)

    print(doc.page_content)

Source 1
major banks and top retailers in Australia has shown me that a solid explainable model can add more value than a complex solution, especially in regulated environments where models need to be explained to different teams to assess their feasibility and risk. For this reason, I deep dive into the data to fully understand the different data sources, how are they connected, and how do they explain customer’s behaviours. For example, in one of my engagements with a big 4 bank, I produced a Machine
Source 2
Peer mentor program at Monash University, Melbourne – AUS

Feb 2020 – May 2020

Mentored new students and participated in the Leap into Leadership program for the development of the following skills:

Foundations of leadership

Communicate with impact

 LANGUAGE SKILLS . 

ENGLISH Fluent

SPANISH Fluent

GERMAN Basic
Source 3
given me the basis for focusing on highly relevant, commercially sounding insights and modelling. I have advanced proficiency using technical tools like SQ

## Test hallucination resistance

In [26]:
# update history of the conversation
history.append(HumanMessage(content=question))
history.append(AIMessage(content=answer))

In [27]:
question = "What machine learning projects did the candidate work on at Google?"
answer, sources = answer_question(question, history=history)

print(answer)

The provided information does not mention any machine learning projects David worked on at Google. It does describe projects in banking, retail, and at APNA, including customer segmentation, churn prediction, store clustering, classification models, and PySpark proof-of-concepts.


## Conclusion

### Decision

Use:

- System prompt defining assistant behavior.
- Conversation history.
- Grounded Retrieval-Augmented Generation.
- Explicit refusal for unsupported questions.

### Rationale

- Improves consistency.
- Reduces hallucinations.
- Supports multi-turn conversations.
- Produces recruiter-friendly responses.

### Next Step

Validate the complete MVP against the product requirements before building the Gradio application.